# Trump Tweets Classification - Feature Engineering

This notebook implements comprehensive feature engineering for the Trump tweets classification task.

## Objectives
1. Load and preprocess the tweet data
2. Extract text-based features (TF-IDF, n-grams)
3. Extract stylistic features (capitalization, punctuation patterns)
4. Extract temporal features (hour, day patterns)
5. Extract metadata features (length, hashtags, mentions)
6. Combine and save all features for model training

In [1]:
# Import required libraries
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

# Import our preprocessing modules
from preprocessing.data_loader import TweetDataLoader
from preprocessing.text_cleaner import TweetTextCleaner
from preprocessing.feature_extractor import extract_features, get_feature_names

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

ImportError: cannot import name 'TweetDataLoader' from 'preprocessing.data_loader' (/Users/eyalbenbarouch/Documents/GitHub/Trump-Tweets-Classafication/experiments/../src/preprocessing/data_loader.py)

## 1. Load and Preprocess Data

In [ ]:
# Load the training data
data_loader = TweetDataLoader()
df = data_loader.load_data('../data/trump_train.tsv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Extract text features using different configurations (conservative for optimal performance)
text_feature_configs = {
    'unigrams': {'ngram_range': (1, 1), 'max_features': 200, 'min_df': 3},
    'bigrams': {'ngram_range': (1, 2), 'max_features': 300, 'min_df': 3},
    'trigrams': {'ngram_range': (1, 3), 'max_features': 400, 'min_df': 3}
}

text_features = {}

for config_name, config in text_feature_configs.items():
    print(f"\\nExtracting {config_name} features...")
    
    # Create TF-IDF vectorizer
    tfidf = TfidfVectorizer(
        max_features=config['max_features'],
        ngram_range=config['ngram_range'],
        min_df=config['min_df'],
        stop_words='english',
        lowercase=True,
        strip_accents='ascii'
    )
    
    # Fit and transform
    features = tfidf.fit_transform(df['cleaned_text'])
    
    print(f"Feature matrix shape: {features.shape}")
    print(f"Feature density: {features.nnz / (features.shape[0] * features.shape[1]):.4f}")
    print(f"Samples per feature: {features.shape[0] / features.shape[1]:.1f}")
    
    # Store features and vectorizer
    text_features[config_name] = {
        'features': features.toarray(),
        'vectorizer': tfidf,
        'feature_names': tfidf.get_feature_names_out()
    }
    
    # Show top features by class
    feature_means_trump = features[df['label'] == 0].mean(axis=0).A1
    feature_means_staffer = features[df['label'] == 1].mean(axis=0).A1
    
    # Top Trump features
    top_trump_idx = np.argsort(feature_means_trump)[-10:]
    print(f"\\nTop {config_name} features for Trump:")
    for idx in reversed(top_trump_idx):
        print(f"  {tfidf.get_feature_names_out()[idx]}: {feature_means_trump[idx]:.4f}")
    
    # Top Staffer features
    top_staffer_idx = np.argsort(feature_means_staffer)[-10:]
    print(f"\\nTop {config_name} features for Staffer:")
    for idx in reversed(top_staffer_idx):
        print(f"  {tfidf.get_feature_names_out()[idx]}: {feature_means_staffer[idx]:.4f}")

In [ ]:
# Clean the text data
text_cleaner = TweetTextCleaner()

print("Before cleaning:")
print(df['tweet_text'].iloc[0])

# Apply text cleaning
df['cleaned_text'] = df['tweet_text'].apply(text_cleaner.clean_text)

print("\nAfter cleaning:")
print(df['cleaned_text'].iloc[0])

# Check for any empty texts after cleaning
empty_texts = df['cleaned_text'].str.strip() == ''
print(f"\nNumber of empty texts after cleaning: {empty_texts.sum()}")

# Remove empty texts if any
if empty_texts.sum() > 0:
    df = df[~empty_texts].reset_index(drop=True)
    print(f"Dataset shape after removing empty texts: {df.shape}")

## 2. Extract Text-based Features (TF-IDF)

In [ ]:
# Extract text features using different configurations
text_feature_configs = {
    'unigrams': {'ngram_range': (1, 1), 'max_features': 3000},
    'bigrams': {'ngram_range': (1, 2), 'max_features': 5000},
    'trigrams': {'ngram_range': (1, 3), 'max_features': 7000}
}

text_features = {}

for config_name, config in text_feature_configs.items():
    print(f"\nExtracting {config_name} features...")
    
    # Create TF-IDF vectorizer
    tfidf = TfidfVectorizer(
        max_features=config['max_features'],
        ngram_range=config['ngram_range'],
        stop_words='english',
        lowercase=True,
        strip_accents='ascii'
    )
    
    # Fit and transform
    features = tfidf.fit_transform(df['cleaned_text'])
    
    print(f"Feature matrix shape: {features.shape}")
    print(f"Feature density: {features.nnz / (features.shape[0] * features.shape[1]):.4f}")
    
    # Store features and vectorizer
    text_features[config_name] = {
        'features': features.toarray(),
        'vectorizer': tfidf,
        'feature_names': tfidf.get_feature_names_out()
    }
    
    # Show top features by class
    feature_means_trump = features[df['label'] == 0].mean(axis=0).A1
    feature_means_staffer = features[df['label'] == 1].mean(axis=0).A1
    
    # Top Trump features
    top_trump_idx = np.argsort(feature_means_trump)[-10:]
    print(f"\nTop {config_name} features for Trump:")
    for idx in reversed(top_trump_idx):
        print(f"  {tfidf.get_feature_names_out()[idx]}: {feature_means_trump[idx]:.4f}")
    
    # Top Staffer features
    top_staffer_idx = np.argsort(feature_means_staffer)[-10:]
    print(f"\nTop {config_name} features for Staffer:")
    for idx in reversed(top_staffer_idx):
        print(f"  {tfidf.get_feature_names_out()[idx]}: {feature_means_staffer[idx]:.4f}")

## 3. Extract Stylistic Features

In [ ]:
# Extract stylistic features using our feature extractor
print("Extracting stylistic features...")

# Use original tweet text for stylistic features (not cleaned)
stylistic_result = extract_features(
    df, 
    feature_types=['stylistic'], 
    text_column='tweet_text'
)

stylistic_features = stylistic_result['stylistic']
print(f"Stylistic features shape: {stylistic_features.shape}")

# Create DataFrame for analysis
stylistic_feature_names = [
    'char_count', 'word_count', 'avg_word_length', 'caps_count', 'caps_ratio', 
    'all_caps_words', 'exclamation_count', 'question_count', 'period_count', 
    'comma_count', 'ellipsis_count', 'hashtag_count', 'mention_count', 
    'url_count', 'emoticon_count'
]

stylistic_df = pd.DataFrame(stylistic_features, columns=stylistic_feature_names)
stylistic_df['label'] = df['label']

print("\nStylistic features statistics:")
print(stylistic_df.describe())

In [ ]:
# Analyze stylistic features by author
print("Stylistic Features by Author:")
print("="*50)

for label, name in [(0, 'Trump'), (1, 'Staffer')]:
    subset = stylistic_df[stylistic_df['label'] == label]
    print(f"\n{name}:")
    for feature in stylistic_feature_names[:10]:  # Show first 10 features
        mean_val = subset[feature].mean()
        print(f"  {feature}: {mean_val:.3f}")

# Visualize key stylistic differences
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

key_features = ['caps_ratio', 'exclamation_count', 'hashtag_count', 'mention_count', 'word_count', 'char_count']

for i, feature in enumerate(key_features):
    trump_data = stylistic_df[stylistic_df['label'] == 0][feature]
    staffer_data = stylistic_df[stylistic_df['label'] == 1][feature]
    
    axes[i].hist(trump_data, bins=30, alpha=0.7, label='Trump', color='red', density=True)
    axes[i].hist(staffer_data, bins=30, alpha=0.7, label='Staffer', color='blue', density=True)
    axes[i].set_title(f'{feature.replace("_", " ").title()}')
    axes[i].set_xlabel(feature.replace('_', ' ').title())
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 4. Extract Temporal Features

In [ ]:
# Extract temporal features from timestamps
print("Extracting temporal features...")

# Filter data with valid timestamps
import re
date_pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'
valid_timestamps = df['timestamp'].str.match(date_pattern, na=False)
df_temporal = df[valid_timestamps].copy()

print(f"Valid timestamps: {len(df_temporal)} / {len(df)} ({len(df_temporal)/len(df)*100:.1f}%)")

if len(df_temporal) > 0:
    temporal_result = extract_features(
        df_temporal, 
        feature_types=['temporal'], 
        text_column='tweet_text'
    )
    
    temporal_features = temporal_result['temporal']
    print(f"Temporal features shape: {temporal_features.shape}")
    
    # Create DataFrame for analysis
    temporal_feature_names = [
        'hour', 'day_of_week', 'is_weekend', 'is_morning', 
        'is_afternoon', 'is_evening', 'is_night'
    ]
    
    temporal_df = pd.DataFrame(temporal_features, columns=temporal_feature_names)
    temporal_df['label'] = df_temporal['label'].values
    
    print("\nTemporal features by author:")
    for label, name in [(0, 'Trump'), (1, 'Staffer')]:
        subset = temporal_df[temporal_df['label'] == label]
        print(f"\n{name}:")
        print(f"  Average hour: {subset['hour'].mean():.1f}")
        print(f"  Weekend posting: {subset['is_weekend'].mean():.3f}")
        print(f"  Morning posting: {subset['is_morning'].mean():.3f}")
        print(f"  Evening posting: {subset['is_evening'].mean():.3f}")
else:
    temporal_features = None
    temporal_df = None
    print("No valid timestamps found for temporal feature extraction.")

## 5. Feature Scaling and Normalization

In [ ]:
# Scale stylistic and temporal features
scaler = StandardScaler()

# Scale stylistic features
stylistic_features_scaled = scaler.fit_transform(stylistic_features)
print(f"Stylistic features scaled shape: {stylistic_features_scaled.shape}")

# Scale temporal features if available
if temporal_features is not None:
    temporal_scaler = StandardScaler()
    temporal_features_scaled = temporal_scaler.fit_transform(temporal_features)
    print(f"Temporal features scaled shape: {temporal_features_scaled.shape}")
else:
    temporal_features_scaled = None
    temporal_scaler = None

# Note: TF-IDF features are already normalized

## 6. Combine Features and Create Different Feature Sets

In [ ]:
# Create different feature combinations for experimentation
feature_sets = {}

# 1. Text-only features (different n-gram configurations)
for config_name, config_data in text_features.items():
    feature_sets[f'text_{config_name}'] = {
        'features': config_data['features'],
        'labels': df['label'].values,
        'description': f'TF-IDF {config_name} features only'
    }

# 2. Stylistic features only
feature_sets['stylistic'] = {
    'features': stylistic_features_scaled,
    'labels': df['label'].values,
    'description': 'Stylistic features only'
}

# 3. Combined text + stylistic
for config_name, config_data in text_features.items():
    combined_features = np.hstack([
        config_data['features'],
        stylistic_features_scaled
    ])
    
    feature_sets[f'text_{config_name}_stylistic'] = {
        'features': combined_features,
        'labels': df['label'].values,
        'description': f'TF-IDF {config_name} + stylistic features'
    }

# 4. Add temporal features if available
if temporal_features_scaled is not None:
    # Temporal only
    feature_sets['temporal'] = {
        'features': temporal_features_scaled,
        'labels': df_temporal['label'].values,
        'description': 'Temporal features only'
    }
    
    # All features combined (for subset with valid timestamps)
    # Need to align indices
    valid_indices = df.index[valid_timestamps]
    
    for config_name, config_data in text_features.items():
        text_subset = config_data['features'][valid_indices]
        stylistic_subset = stylistic_features_scaled[valid_indices]
        
        all_combined = np.hstack([
            text_subset,
            stylistic_subset,
            temporal_features_scaled
        ])
        
        feature_sets[f'all_{config_name}'] = {
            'features': all_combined,
            'labels': df_temporal['label'].values,
            'description': f'All features: TF-IDF {config_name} + stylistic + temporal'
        }

# Print summary of feature sets
print("Feature Sets Summary:")
print("="*60)
for name, data in feature_sets.items():
    print(f"{name:25} | Shape: {data['features'].shape} | {data['description']}")

## 7. Save Features and Extractors

In [ ]:
# Create processed data directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save all feature sets
print("Saving feature sets...")
for name, data in feature_sets.items():
    filename = f'../data/processed/features_{name}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved: {filename}")

# Save extractors and scalers
extractors = {
    'text_vectorizers': {name: data['vectorizer'] for name, data in text_features.items()},
    'stylistic_scaler': scaler,
    'temporal_scaler': temporal_scaler,
    'text_cleaner': text_cleaner,
    'data_loader': data_loader
}

with open('../data/processed/extractors.pkl', 'wb') as f:
    pickle.dump(extractors, f)
print("Saved: extractors.pkl")

# Save feature names for interpretability
feature_names = {
    'stylistic': stylistic_feature_names,
    'temporal': temporal_feature_names if temporal_features is not None else None,
    'text': {name: data['feature_names'].tolist() for name, data in text_features.items()}
}

with open('../data/processed/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print("Saved: feature_names.pkl")

# Save processed dataframe
df_processed = df.copy()
df_processed.to_csv('../data/processed/tweets_processed.csv', index=False)
print("Saved: tweets_processed.csv")

if temporal_df is not None:
    df_temporal.to_csv('../data/processed/tweets_temporal.csv', index=False)
    print("Saved: tweets_temporal.csv")

## 8. Feature Analysis Summary

In [ ]:
# Create feature analysis summary
print("FEATURE ENGINEERING SUMMARY")
print("="*60)
print(f"Total tweets processed: {len(df)}")
print(f"Tweets with valid timestamps: {len(df_temporal) if temporal_df is not None else 0}")
print(f"Trump tweets: {(df['label'] == 0).sum()}")
print(f"Staffer tweets: {(df['label'] == 1).sum()}")

print("\nFEATURE TYPES:")
print("-" * 30)
print(f"Text features (TF-IDF):")
for config_name, config_data in text_features.items():
    print(f"  - {config_name}: {config_data['features'].shape[1]} features")

print(f"\nStylistic features: {stylistic_features.shape[1]} features")
if temporal_features is not None:
    print(f"Temporal features: {temporal_features.shape[1]} features")
else:
    print("Temporal features: Not available (no valid timestamps)")

print(f"\nFEATURE SETS CREATED: {len(feature_sets)}")
print("-" * 30)
for name, data in feature_sets.items():
    print(f"{name}: {data['features'].shape}")

print("\nFILES SAVED:")
print("-" * 30)
print("- Feature sets: features_*.pkl")
print("- Extractors and scalers: extractors.pkl")
print("- Feature names: feature_names.pkl")
print("- Processed data: tweets_processed.csv")
if temporal_df is not None:
    print("- Temporal data: tweets_temporal.csv")

print("\nREADY FOR MODEL TRAINING!")
print("="*60)